In [2]:
import torch
import torch.nn as nn
import math


d_head = number of attention heads per block

d_emb = dimension of the embedding vector for a token
d_kq = dimensions of the key/query spave

let each embedding "E" be a column vector of size d_emb

W_q and W_k should have dimensions of d_kq x d_emb

In [3]:
#each input is a sequence of tokens of length seq_len
#we have a batch size of n inputs
#So the overall input to the model is of dimensions (n, seq_len, d_emb)
#So after the linear layers, Q and K have dimensions of (n, seq_len, d_kq)
#d_v is the output size for the single head of attention for a single token.
#This is not the same as the embedding space since we use the concatenated output of multiple heads of attention to get the final output

class selfAttentionHead(nn.Module):
    def __init__(self, d_emb: int, d_kq: int, d_v):
        super().__init__()
        self.Wq = nn.Linear(d_emb, d_kq, bias=False)
        self.Wk = nn.Linear(d_emb, d_kq, bias=False)
        self.Wv = nn.Linear(d_emb, d_v)

        self.d_k = d_kq
        self.d_emb = d_emb
        self.d_v = d_v

    def forward(self,x: torch.Tensor):
        #get query, key, and value vectors
        #dimensions of (n, seq_len, d_k)
        Q = self.Wq(x)
        K = self.Wk(x)

        #dimensions of (n, seq_len, d_emb)
        V = self.Wv(x)

        #compute attention weights
        #dimensions of (n, seq_len, seq_len)
        scores = torch.bmm(Q, K.transpose(1,2))
        scores = scores/(self.d_k ** 0.5)

        attentionWeights = nn.functional.softmax(scores, dim=-1)
        print(attentionWeights.size())


        output = torch.bmm(attentionWeights, V)

        return output
        



In [4]:
#here we are initializing a random input to verify the output dimensions of the single head of attention
x = torch.rand(2,4,128)

model = selfAttentionHead(128, 32, 64)
output = model.forward(x)

print(output.size())

torch.Size([2, 4, 4])
torch.Size([2, 4, 64])


In [3]:
#An attention block uses multiple self attention heads in parallel
#We can replicate this by making some changes to our matrix operations
#here for simplicity we will set d_v = d_k = d_emb / num_heads
class MultiHeadedAttentionBlock(nn.Module):
    def __init__(self, d_emb: int, num_heads: int, masked: bool):
        super().__init__()

        if d_emb % num_heads != 0:
            raise ValueError("Embedding dimension must be divisible by number of heads")

        self.d_k = d_emb // num_heads

        #we are stacking each weight matrix of size d_emb x d_k next to each other to create one matrix that represents all the heads
        self.W_q = nn.Linear(d_emb, d_emb, bias=False)
        self.W_k = nn.Linear(d_emb, d_emb, bias=False)
        self.W_v = nn.Linear(d_emb, d_emb, bias=False)
        self.W_final = nn.Linear(d_emb,d_emb, bias=False)

        self.num_heads = num_heads
        self.d_emb = d_emb


        #define if we are using masked attention
        self.masked = masked

    def forward(self, x: torch.Tensor, padding_mask = None):
        batch_size, seq_len, d_emb = x.size()

        Q = self.W_q(x)
        K = self.W_k(x)
        V = self.W_v(x)

        #torch.matmul works on the last two dimensions of the tensor 
        #Because of this we want the last two dimensions to be the sequence length and d_k
        #The view operation seperates the heads from eachother into a new dimension
        #Transposing leaves the last two dimensions as seq_len and d_k

        Q = Q.view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1,2)
        K = K.view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1,2)
        V = V.view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1,2)
        
        #Scores is a matrix of size seq_len x seq_len
        scores = torch.matmul(Q, K.transpose(-2,-1))/(self.d_k ** 0.5)

        #Now we need to mask out future tokens if we are creating a masked block
        if(self.masked):
            mask = torch.triu(torch.ones(seq_len, seq_len, device=x.device), diagonal=1).bool()
        #applying mask
            masked_scores = scores.masked_fill(mask, float('-inf'))
            scores = masked_scores

        
        if padding_mask is not None:
            scores = scores.masked_fill(padding_mask[:, None, None, :],float("-inf"))
        
        attentionWeights = torch.softmax(scores, dim=-1)

        context = torch.matmul(attentionWeights, V)

        #undoing the transposing and reshaping we did to get a matrix of (batch_size, seq_len, model)
        out = context.transpose(1,2).contiguous().view(batch_size, seq_len, self.num_heads * self.d_k)
        

        #In the original paper attention is all you need, they use a final linear layer after concatenation
        output = self.W_final(out)

        return output




In [4]:
class EncoderLayer(nn.Module):
    def __init__(self, d_emb, num_heads, d_ff):
        super().__init__()

        self.attention = MultiHeadedAttentionBlock(d_emb, num_heads, masked=False)
        self.dropout1 = nn.Dropout(0.1)
        self.dropout2 = nn.Dropout(0.1)


        self.FFN = nn.Sequential(
            nn.Linear(d_emb, d_ff),
            nn.ReLU(),
            nn.Linear(d_ff, d_emb)
        )

        self.addNorm1 = nn.LayerNorm(d_emb)
        self.addNorm2 = nn.LayerNorm(d_emb)

    def forward(self, x, padding_mask=None):

        xNorm = self.addNorm1(x)

        attention_out = self.attention(xNorm, padding_mask=padding_mask)

        sum_attention = x + self.dropout1(attention_out)

        attentionNorm = self.addNorm2(sum_attention)

        output = self.FFN(self.dropout2(attentionNorm)) + sum_attention

        return output

In [5]:
class CrossAttentionBlock(nn.Module):
    def __init__(self,d_emb, num_heads):
        super().__init__()
        if d_emb % num_heads != 0:
            raise ValueError("Embedding dimension must be divisible by number of heads")
        
        self.d_k = d_emb // num_heads

        self.W_q = nn.Linear(d_emb, d_emb, bias=False)
        self.W_k = nn.Linear(d_emb, d_emb, bias=False)
        self.W_v = nn.Linear(d_emb, d_emb, bias=False)
        self.W_final = nn.Linear(d_emb,d_emb, bias=False)
        
        self.num_heads = num_heads
        self.d_emb = d_emb


    def forward(self, x: torch.Tensor, y: torch.Tensor, padding_mask=None):
        batch_size, seq_x, d_emb = x.size()
        batch_size, seq_y, d_emb = y.size()

        Q = self.W_q(x)
        K = self.W_k(y)
        V = self.W_v(y)

        Q = Q.view(batch_size, seq_x, self.num_heads, self.d_k).transpose(1,2)
        K = K.view(batch_size, seq_y, self.num_heads, self.d_k).transpose(1,2)
        V = V.view(batch_size, seq_y, self.num_heads, self.d_k).transpose(1,2)

        #for cross attention, the scores is a matrix of seq_x by seq_y
        scores = torch.matmul(Q, K.transpose(-1,-2))/(self.d_k ** 0.5)

        #Masks out padding tokens
        if padding_mask is not  None:
            scores = scores.masked_fill(padding_mask[:, None, None, :], float("-inf"))

        attenWeights = torch.softmax(scores, dim=-1)

        out = torch.matmul(attenWeights, V)

        out = out.transpose(1,2,).contiguous().view(batch_size, seq_x, self.d_k * self.num_heads)

        output = self.W_final(out)
        
        return output

In [6]:
class DecoderLayer(nn.Module):
    def __init__(self, d_emb, num_heads, d_ff):
        super().__init__()
        self.d_emb = d_emb
        self.num_heads = num_heads
        self.d_ff = d_ff

        self.masked_attention = MultiHeadedAttentionBlock(d_emb, num_heads, masked=True)
        self.cross_attention = CrossAttentionBlock(d_emb, num_heads)

        self.FFN = nn.Sequential(
                                nn.Linear(d_emb, d_ff),
                                nn.ReLU(),
                                nn.Linear(d_ff, d_emb)
        )

        self.addNorm1 = nn.LayerNorm(d_emb)
        self.addNorm2 = nn.LayerNorm(d_emb)
        self.addNorm3 = nn.LayerNorm(d_emb)

        self.dropout1 = nn.Dropout(0.1)
        self.dropout2 = nn.Dropout(0.1)
        self.dropout3 = nn.Dropout(0.1)

    def forward(self, x, y, target_padding_mask=None, source_padding_mask=None):

        xNorm = self.addNorm1(x)
    
        mask_out = self.masked_attention(xNorm, padding_mask=target_padding_mask)

        sum_mask_out = x + self.dropout1(mask_out)

        mask_out = self.addNorm2(sum_mask_out)

        cross_out = self.cross_attention(mask_out, y, padding_mask=source_padding_mask)

        sum_cross_out = self.dropout2(cross_out) + sum_mask_out

        cross_out = self.addNorm3(sum_cross_out)

        ffn_out = self.FFN(cross_out) + sum_cross_out

        return ffn_out


In [7]:
class Encoder(nn.Module):
    def __init__(self, d_emb, num_heads, d_ff, num_layers):
        super().__init__()
        self.d_emb = d_emb
        self.heads = num_heads

        self.num_layers = num_layers

        self.layers = nn.ModuleList([EncoderLayer(d_emb, num_heads, d_ff) for _ in range(num_layers) ])
        self.layerNorm = nn.LayerNorm(d_emb)

    def forward(self, x, padding_mask=None):
        for layer in self.layers:
            x = layer(x, padding_mask)

        x = self.layerNorm(x)
        return x

In [8]:
class Decoder(nn.Module):
    def __init__(self, d_emb, num_heads, d_ff, num_layers):
        super().__init__()
        self.num_layers = num_layers
        self.layers = nn.ModuleList([DecoderLayer(d_emb, num_heads, d_ff) for _ in range(num_layers)])

        self.layerNorm = nn.LayerNorm(d_emb)

    def forward(self, x, y, target_padding_mask=None, source_padding_mask=None):
        for layer in self.layers:
            x = layer(x, y, target_padding_mask, source_padding_mask)

        x = self.layerNorm(x)
        return x

In [9]:
# Generated from ChatGPT - Based on positional encoding scheme from Attention is All You Need
class PositionalEncoding(nn.Module):
    def __init__(self, d_emb, max_len=5000):
        super().__init__()

        position = torch.arange(max_len).unsqueeze(1)

        div_term = torch.exp(
            torch.arange(0, d_emb, 2) *
            (-math.log(10000.0) / d_emb)
        )

        pe = torch.zeros(max_len, d_emb)

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(
            position * div_term[:pe[:, 1::2].shape[1]]
        )

        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x):
        seq_len = x.size(1)
        return x + self.pe[:, :seq_len, :]

        

In [10]:
class EmbeddingLayer(nn.Module):
    def __init__(self, d_emb, vocab_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_emb)
        self.positional_encoding = PositionalEncoding(d_emb)
        self.d_emb = d_emb

    def forward(self, x):
        x = self.embedding(x)
        x = math.sqrt(self.d_emb) * x
        x = self.positional_encoding(x)
        return x

In [11]:
class Transformer(nn.Module):
    def __init__(self, d_emb, d_ff, target_vocab_size, source_vocab_size, num_heads, num_layers_encoder, num_layers_decoder):
        super().__init__()
        self.decoder = Decoder(d_emb, num_heads, d_ff, num_layers_decoder)
        self.encoder = Encoder(d_emb, num_heads, d_ff, num_layers_encoder)

        self.target_embedding = EmbeddingLayer(d_emb, target_vocab_size)
        self.source_embedding = EmbeddingLayer(d_emb, source_vocab_size)
        self.output = nn.Linear(d_emb, target_vocab_size)

        self.d_emb = d_emb
        self.d_ff = d_ff
        self.num_heads = num_heads
        self.encoder_layers = num_layers_encoder
        self.decoder_layers = num_layers_decoder

        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.xavier_normal_(module.weight)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.xavier_normal_(module.weight)
        elif isinstance(module, nn.LayerNorm):
            nn.init.ones_(module.weight)
            nn.init.zeros_(module.bias)

    def encode(self, source_in, source_padding_mask=None):
        source_embeddings = self.source_embedding(source_in)
        encoder_output = self.encoder(source_embeddings, source_padding_mask)
        return encoder_output
    
    def decode(self, target_in, encoder_output, target_padding_mask=None, source_padding_mask=None):
        target_embeddings = self.target_embedding(target_in)
        decoder_output = self.decoder(target_embeddings, encoder_output, target_padding_mask, source_padding_mask)
        logits = self.output(decoder_output)
        return logits

    def forward(self, source_in, target_in, target_padding_mask = None, source_padding_mask = None):
        encoder_output = self.encode(source_in, source_padding_mask)
        output = self.decode(target_in, encoder_output, target_padding_mask, source_padding_mask)
        return output

    

In [12]:
from custom_tokenizer import byte_pair_encoding, generate_vocab_dictionary, tokenize_corpus, encode, decode, normalize
translation_pairs = []
with open("spa-eng/spa.txt", "r", encoding="utf-8") as f:
    for line in f:
        english, spanish, attributions = line.rstrip("\n").split("\t")
        translation_pairs.append((english, spanish))

english = [item[0] for item in translation_pairs]
spanish = [item[1] for item in translation_pairs]


In [66]:
english_freq, english_rules, english_vocab = byte_pair_encoding(english, 8000)
spanish_freq, spanish_rules, spanish_vocab = byte_pair_encoding(spanish, 8000)

In [103]:
english_vocab_dict = generate_vocab_dictionary(english_vocab)
spanish_vocab_dict = generate_vocab_dictionary(spanish_vocab)

In [91]:
english_tokens = []
spanish_tokens = []

english_ids = []
spanish_ids = []

english_tokens = tokenize_corpus(english, english_rules)
spanish_tokens = tokenize_corpus(spanish, spanish_rules)

In [108]:
for tokens in english_tokens:
    english_ids.append(encode(tokens, english_vocab_dict))
for tokens in spanish_tokens:
    spanish_ids.append(encode(tokens, spanish_vocab_dict))

In [117]:
english_tokenizer = {
    "vocab_dict":english_vocab_dict,
    "merge_rules":english_rules,
    "encoded_corpus": english_ids
}

spanish_tokenizer = {
    "vocab_dict":spanish_vocab_dict,
    "merge_rules":spanish_rules,
    "encoded_corpus": spanish_ids
}

torch.save(english_tokenizer, "saved_tokenizers/english_tokenizer.pt")
torch.save(spanish_tokenizer, "saved_tokenizers/spanish_tokenizer.pt")

In [13]:
from torch.nn.utils.rnn import pad_sequence

def pad_data(batch):
    english_batch, spanish_batch = zip(*batch)

    english_batch = [torch.tensor(seq, dtype=torch.long) for seq in english_batch]
    spanish_batch = [torch.tensor(seq, dtype=torch.long) for seq in spanish_batch]

    english_batch = pad_sequence(english_batch, batch_first=True, padding_value=0)
    spanish_batch = pad_sequence(spanish_batch, batch_first=True, padding_value=0)

    return english_batch, spanish_batch

In [216]:
from torch.utils.data import DataLoader, random_split


BATCH_SIZE = 64

encoded_pairs = list(zip(english_ids, spanish_ids))

dataset = encoded_pairs

train_size = int(0.8 * len(dataset))
val_size = int(0.1 * len(dataset))
test_size = len(dataset) - train_size - val_size

generator = torch.Generator().manual_seed(42)

train_dataset, val_dataset, test_dataset = random_split(
    dataset,
    [train_size, val_size, test_size],
    generator=generator
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=pad_data)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=pad_data)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=pad_data)

In [14]:
D_EMB = 512
D_FF = 2048
NUM_HEADS = 8
NUM_ENCODER_LAYERS = 6
NUM_DECODER_LAYERS = 6



model = Transformer(d_emb=D_EMB, 
                    d_ff=D_FF, 
                    target_vocab_size=len(spanish_vocab_dict), 
                    source_vocab_size=len(english_vocab_dict), 
                    num_heads=NUM_HEADS, 
                    num_layers_encoder=NUM_ENCODER_LAYERS, 
                    num_layers_decoder=NUM_DECODER_LAYERS)

device = torch.device(
    "mps" if torch.backends.mps.is_available() else "cpu"
)

model = model.to(device)


criterion = nn.CrossEntropyLoss(ignore_index=0)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001, betas=(0.9, 0.98))
scheduler = torch.optim.lr_scheduler.StepLR(optimizer=optimizer, gamma=0.1, step_size=8)

NUM_EPOCHS = 40
best_val_loss = float('inf')

for epoch in range(NUM_EPOCHS):
    model.train()
    running_loss = 0.0

    for english_batch, spanish_batch in train_loader:

        english_batch = english_batch.to(device)
        spanish_batch = spanish_batch.to(device)

        decoder_input = spanish_batch[:, :-1]
        targets = spanish_batch[:, 1:]

        source_padding_mask = english_batch == 0
        target_padding_mask = decoder_input == 0

        
        optimizer.zero_grad(set_to_none=True)
        outputs = model(english_batch, decoder_input, target_padding_mask, source_padding_mask)

        loss = criterion(outputs.reshape(-1, outputs.size(-1)),targets.reshape(-1))

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

    avg_train_loss = running_loss/len(train_loader)

    model.eval()
    running_val_loss = 0.0
    with torch.no_grad():
        for english_batch, spanish_batch in val_loader:
            english_batch = english_batch.to(device)
            spanish_batch = spanish_batch.to(device)

            decoder_input = spanish_batch[:, :-1]
            targets = spanish_batch[:, 1:]
        
            source_padding_mask = english_batch == 0
            target_padding_mask = decoder_input == 0
        
            outputs = model(english_batch, decoder_input, target_padding_mask, source_padding_mask)

            loss = criterion(outputs.reshape(-1, outputs.size(-1)),targets.reshape(-1))

            running_val_loss += loss.item()

    avg_val_loss = running_val_loss/len(val_loader)
    print(f"Epoch [{epoch+1}/{NUM_EPOCHS}] | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")

    scheduler.step()

    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save(model.state_dict(), "best_model_weights.pt")


        

NameError: name 'spanish_vocab_dict' is not defined

In [15]:
english_tokenizer = torch.load("saved_tokenizers/english_tokenizer.pt")
spanish_tokenizer = torch.load("saved_tokenizers/spanish_tokenizer.pt")

english_vocab_dict = english_tokenizer["vocab_dict"]
english_rules = english_tokenizer["merge_rules"]

spanish_vocab_dict = spanish_tokenizer["vocab_dict"]
spanish_rules = spanish_tokenizer["merge_rules"]



model = Transformer(d_emb=D_EMB, 
                    d_ff=D_FF, 
                    target_vocab_size=len(spanish_vocab_dict), 
                    source_vocab_size=len(english_vocab_dict), 
                    num_heads=NUM_HEADS, 
                    num_layers_encoder=NUM_ENCODER_LAYERS, 
                    num_layers_decoder=NUM_DECODER_LAYERS)

state_dict = torch.load("best_model_weights.pt")


model.load_state_dict(state_dict)

model.eval()

Transformer(
  (decoder): Decoder(
    (layers): ModuleList(
      (0-5): 6 x DecoderLayer(
        (masked_attention): MultiHeadedAttentionBlock(
          (W_q): Linear(in_features=512, out_features=512, bias=False)
          (W_k): Linear(in_features=512, out_features=512, bias=False)
          (W_v): Linear(in_features=512, out_features=512, bias=False)
          (W_final): Linear(in_features=512, out_features=512, bias=False)
        )
        (cross_attention): CrossAttentionBlock(
          (W_q): Linear(in_features=512, out_features=512, bias=False)
          (W_k): Linear(in_features=512, out_features=512, bias=False)
          (W_v): Linear(in_features=512, out_features=512, bias=False)
          (W_final): Linear(in_features=512, out_features=512, bias=False)
        )
        (FFN): Sequential(
          (0): Linear(in_features=512, out_features=2048, bias=True)
          (1): ReLU()
          (2): Linear(in_features=2048, out_features=512, bias=True)
        )
        (add

In [19]:
input_text = ["the cat is short"]
spanish_translation = [1]
tokenized_input = tokenize_corpus(input_text, english_rules)

print(tokenized_input)


encoded_input = torch.tensor(encode(tokenized_input[0], english_vocab_dict), dtype=torch.long).unsqueeze(0)

pred_token = None
seq_len = 0
max_len = 100

with torch.no_grad(): 
    encoder_output = model.encode(encoded_input)

    while pred_token != 2 and seq_len <= max_len:

        decoder_input = torch.tensor(
            [spanish_translation],
            dtype=torch.long,
            device=encoded_input.device
        )

        decoder_output = model.decode(decoder_input, encoder_output=encoder_output)
        next_token_logits = decoder_output[:, -1, :]

        # Select the token with the highest logit
        pred_token = next_token_logits.argmax(dim=-1).item()

        spanish_translation.append(pred_token)
        seq_len += 1

translation = decode(spanish_translation, spanish_vocab_dict)
print(translation)

[['the</w>', 'cat</w>', 'is</w>', 'short</w>']]
el gato está lleno de cáncer.
